# Ball Detector Baseline - Shooting Drill Footage

**Goal:** Evaluate baseline ball detector models on high quality shooting drill footage shot from several different angles.

**Videos:** high quality shooting drill footage shot from several different angles

Camera angles observed (*{horizontal to subject}-{height}-{distance}*):

*NOTE: low height is ~hip level, mid and high is drone level*
- behind-high-mid
- behind-mid-far
- behind-mid-mid
- behind-low-mid
- behind-low-close
- side-low-far

**Models evaluated:**
- `yolo11n` fine-tuned on Roboflow `football-ball-detection` dataset (v4)
- FootAndBall - pretrained, broadcast-trained weights

*Note: yolo11n is chosen over the yolo11(l)(m) models out of consideration for computational costs*

**Evaluation approach (no ground truth labels):**
- Detection rate: frames with >= 1 ball detection / total frames
- Track continuity: mean consecutive-frame streak before track loss
- Visual inspection per clip per camera angle
- Annotated output video per model per clip

*Note: models should be re-evaluated (and fine-tuned/trained) later on custom labelled data*

## 0. Setup

In [1]:
import os
import sys

from IPython.display import clear_output

REPOURL = "https://github.com/WillEdgington/football-kick-tracker.git"
REPODIR = "football-kick-tracker"

if not os.path.exists(REPODIR):
    !git clone {REPOURL}
else:
    !git -C {REPODIR} pull origin main

%pip install -e {REPODIR} -q

projectRoot = os.path.join(os.getcwd(), REPODIR)
if projectRoot not in sys.path:
    sys.path.insert(0, projectRoot)

if not os.path.exists("FootAndBall"):
    !git clone https://github.com/jac99/FootAndBall.git
    %pip install -q -r FootAndBall/requirements.txt

fbPath = os.path.join(os.getcwd(), "FootAndBall")
if fbPath not in sys.path:
    sys.path.append(fbPath)

clear_output()

print("Environment Setup Complete.")
print(f"Project Root added to sys.path: {projectRoot}")

In [9]:
import os
import sys
from pathlib import Path
from typing import Dict, List, Tuple

import cv2
import numpy as np
import torch
from google.colab import drive
from roboflow import Roboflow
from ultralytics import YOLO

from ball.config import ANNOTATEDBALLVIDEOSDIR
from utils.config import RAWTRAININGVIDEOSDIR, SESSIONSDIR
from utils.io import loadJSON, saveJSON

drive.mount('/content/drive')
driveRepoDir = Path("/content/drive/MyDrive/football-kick-tracker")

sessionsDir: Path = driveRepoDir / SESSIONSDIR / "ball"
annotatedDir: Path = driveRepoDir / ANNOTATEDBALLVIDEOSDIR
rawVideosDir: Path = driveRepoDir / RAWTRAININGVIDEOSDIR
resultsPath: Path = sessionsDir / "ball_baseline_results.json"
resultsCachePath: Path = sessionsDir / "ball_baseline_video_cache.json"

print(f"torch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Clip Paths

In [10]:
videoPaths = sorted(
    p for suf in ("*.mp4", "*.mov")
    for p in rawVideosDir.glob(suf)
    if p.stem.startswith("shooting-")
)
assert len(videoPaths) > 0, f'No clips with prefix "shooting-" found in {rawVideosDir}'
print(f"Found {len(videoPaths)} clips:")
for p in videoPaths:
    print(f"  {p.name}")

## 3. Fine-tune YOLO11n on Roboflow Football-Ball Dataset

Using `yolo11n` (nano)

Ball-only, single-class detection head.

In [3]:
bestWeights = driveRepoDir / "models/yolo11n_ball/weights/best.pt"

if bestWeights.exists():
    print(
        f"Fine-tuned weights already on Drive at {bestWeights}\n"
        "skipping data loading..."
    )
else:
    APIKEY = input("Your Roboflow API key here: ")
    print(f"API key: {APIKEY}")
    rf = Roboflow(api_key=APIKEY)
    project = rf.workspace("roboflow-jvuqo").project("football-ball-detection-rejhg")
    dataset = project.version(4).download(
        "yolov11", 
        location="/content/rf_ball_dataset"
    )

    import yaml

    with open("/content/rf_ball_dataset/data.yaml") as f:
        print(yaml.safe_load(f))

In [4]:
import shutil

if bestWeights.exists():
    print(f"Fine-tuned weights already on Drive at {bestWeights}\nskipping training...")
else:
    yoloft = YOLO("yolo11n.pt")
    yoloft.train(
        data="/content/rf_ball_dataset/data.yaml",
        epochs=30,
        imgsz=640,
        batch=16,
        device=device,
        project="/content/runs/ball_ft",
        name="yolo11n_ball",
        exist_ok=True,
        verbose=False,
    )

    src = Path("/content/runs/ball_ft/yolo11n_ball/weights/best.pt")
    assert src.exists(), f"Source not found: {src}"
    bestWeights.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, bestWeights)
    print(f"Fine-tuned weights at: {bestWeights}")

## 4. Load models

In [5]:
import FootAndBall.network.footandball as fabmodel

yoloball = YOLO(str(bestWeights)) # fine-tuned yolo11n candidate

# load FootAndBall model
fabWeightsCandidates = list(Path("/content/FootAndBall/models").glob("*.pth"))
assert len(fabWeightsCandidates) > 0, (
    "No .pth files found in: /content/FootAndBall/models"
)
print(
    f"FootAndBall weight files found, picking first one:"
    f" {fabWeightsCandidates[0]}"
)

fabWeights = fabWeightsCandidates[0]
fab = fabmodel.model_factory("fb1", 'detect')
state = torch.load(fabWeights, map_location=device)
fab.load_state_dict(state)
fab.eval().to(device)
models = [
    ("yolo11n_ft", yoloball),
    ("footandball", fab),
]
modelListString = "\n  ".join([name for name, _ in models])
print(f"Models loaded:\n  {modelListString}")

## 5. Inference Helpers

In [6]:
from ultralytics.engine.results import Results

CONFYOLO = 0.25
CONFFAB = 0.5
COLOURS = {
    "yolo11n_ft": (0, 255, 0),
    "footandball": (255, 100, 0),
}
FABBALLLABEL = 1

def _parseFABFrame(
    model: torch.nn.Module,
    frame: np.ndarray,
    conf: float = CONFFAB,
) -> List[Tuple]:
    h, w = frame.shape[:2]
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    t = torch.from_numpy(
        img.transpose(2, 0, 1)
    ).unsqueeze(0).to(next(model.parameters()).device)
    with torch.no_grad():
        detections = model(t)[0]
    out = []
    boxes  = detections["boxes"]
    labels = detections["labels"]
    scores = detections["scores"]
    for box, label, score in zip(boxes, labels, scores):
        if label != FABBALLLABEL or score < conf:
            continue
        x1, y1, x2, y2 = map(int, box.tolist()) # footandball model x1, y1, x2, y2
        out.append((x1, y1, x2, y2, float(score)))
    return out

def _parseYOLOResult(result: Results) -> List[Tuple]:
    if result.boxes is None or len(result.boxes) == 0:
        return []
    return [
        (*map(int, box.xyxy[0].tolist()), float(box.conf[0]))
        for box in result.boxes
    ]

def _annotateFrame(
    frame: np.ndarray, 
    dets: List[Tuple], 
    colour: Tuple
) -> np.ndarray:
    ann = frame.copy()
    for (x1, y1, x2, y2, c) in dets:
        cv2.rectangle(ann, (x1, y1), (x2, y2), colour, 2)
        cv2.putText(ann, f"{c:.2f}", (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, colour, 1)
    return ann

def _makeWriter(
    path: Path,
    fps: float,
    width: int,
    height: int,
) -> cv2.VideoWriter:
    path.parent.mkdir(parents=True, exist_ok=True)
    return cv2.VideoWriter(
        str(path), 
        cv2.VideoWriter_fourcc(*"mp4v"), 
        fps, 
        (width, height)
    )

def _videoMeta(path: Path) -> Tuple[float, int, int]:
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return fps, width, height, frames

def runOnVideoYOLO(
    model: YOLO,
    videoPath: Path,
    annotatePath: Path,
    colour: Tuple[int, int, int]=COLOURS["yolo11n_ft"],
    conf: float=CONFYOLO,
    logging: bool=True,
) -> List[bool]:
    fps, w, h, _ = _videoMeta(videoPath)
    writer = _makeWriter(annotatePath, fps, w, h) if annotatePath is not None else None
    flags = []
    for i, result in enumerate(model(
        str(videoPath), 
        conf=conf, 
        stream=True, 
        verbose=False
    )):
        if logging:
            print(f"  [YOLO] frame {i + 1}", end="\r")
        dets = _parseYOLOResult(result)
        flags.append(len(dets) > 0)
        if writer is not None:
            writer.write(_annotateFrame(result.orig_img, dets, colour))
    if writer is not None:
        writer.release()
    if logging:
        print()
    return flags

def runOnVideoFAB(
    model: torch.nn.Module,
    videoPath: Path,
    annotatePath: Path|None=None,
    colour: Tuple[int, int, int]=COLOURS["footandball"],
    conf: float=CONFFAB,
    logging: bool=True,
) -> List[bool]:
    fps, w, h, nframes = _videoMeta(videoPath)
    writer = _makeWriter(annotatePath, fps, w, h) if annotatePath is not None else None
    cap = cv2.VideoCapture(str(videoPath))
    flags = []
    
    for i in range(nframes):
        ret, frame = cap.read()
        if not ret:
            break
        if logging:
            print(f"  [FAB] frame {i + 1}", end="\r")
        dets = _parseFABFrame(model, frame, conf)
        flags.append(len(dets) > 0)
        if writer is not None:
            writer.write(_annotateFrame(frame, dets, colour))
    cap.release()
    if writer is not None:
        writer.release()
    if logging:
        print()
    return flags

## 6. Per-clip Evaluation

In [7]:
def trackContinuity(flags: List[bool]) -> float:
    streaks, cur = [], 0
    for f in flags:
        if f:
            cur += 1
            continue
        if cur > 0:
            streaks.append(cur)
        cur = 0
    if cur > 0:
        streaks.append(cur)
    return float(np.mean(streaks)) if streaks else 0.0

def detectionRate(flags: List[bool]) -> float:
    cur = 0
    for f in flags:
        cur += int(f)
    return cur / len(flags)

def _evalVideoYOLO(
    model: YOLO,
    videoPath: Path,
    name: str="yolo11n_ft",
    annotatePath: Path|None=None,
    colour: Tuple[int, int, int]=COLOURS["yolo11n_ft"],
    conf: float=CONFYOLO,
    cache: Dict|None=None,
    cachePath: Path|None=None,
    logging: bool=True,
) -> Dict[str, str|float]:
    cacheKey = f"{name}_{videoPath.name}"
    if (cache is not None and cacheKey in cache) and \
        (annotatePath is None or annotatePath.exists()):
        if logging:
            print(f"[{name}] Skipping {videoPath.name} (cached)")
        return cache[cacheKey]
    
    if logging:
        print(f"[{name}] Processing {videoPath.name}...")
    results = {}

    flags = runOnVideoYOLO(
        model=model,
        videoPath=videoPath,
        annotatePath=annotatePath,
        colour=colour,
        conf=conf,
        logging=logging,
    )

    results["mean_track_continuity"] = trackContinuity(flags=flags)
    results["detection_rate"] = detectionRate(flags=flags)
    results["model"] = name
    
    if cache is not None:
        cache[cacheKey] = results
        if cachePath is not None:
            saveJSON(cache, cachePath)
    return results

def _evalVideoFAB(
    model: torch.nn.Module,
    videoPath: Path,
    name: str="footandball",
    annotatePath: Path|None=None,
    colour: Tuple[int, int, int]=COLOURS["footandball"],
    conf: float=CONFFAB,
    cache: Dict|None=None,
    cachePath: Path|None=None,
    logging: bool=True,
) -> Dict[str, str|float]:
    cacheKey = f"{name}_{videoPath.name}"
    if (cache is not None and cacheKey in cache) and \
        (annotatePath is None or annotatePath.exists()):
        if logging:
            print(f"[{name}] Skipping {videoPath.name} (cached)")
        return cache[cacheKey]
    
    if logging:
        print(f"[{name}] Processing {videoPath.name}...")
    results = {}

    flags = runOnVideoFAB(
        model=model,
        videoPath=videoPath,
        annotatePath=annotatePath,
        colour=colour,
        conf=conf,
        logging=logging,
    )

    results["mean_track_continuity"] = trackContinuity(flags=flags)
    results["detection_rate"] = detectionRate(flags=flags)
    results["model"] = name
    
    if cache is not None:
        cache[cacheKey] = results
        if cachePath is not None:
            saveJSON(cache, cachePath)
    return results

def _getOutputPaths(
    name: str,
    sourcePaths: List[Path],
    outputDir: Path,
) -> List[Path]:
    return [outputDir / name / path.with_suffix(".mp4").name for path in sourcePaths]
    
def evaluateVideosYOLO(
    model: YOLO,
    videoPaths: List[Path],
    conf: float=CONFYOLO,
    name: str="yolo11n_ft",
    annotateDir: Path|None=None,
    colour: Tuple[int, int, int]=COLOURS["yolo11n_ft"],
    cache: Dict|None=None,
    cachePath: Path|None=None,
    logging: bool=True,
) -> Dict[str, str|float]|None:
    annotatePaths = _getOutputPaths(
        name=name,
        sourcePaths=videoPaths,
        outputDir=annotateDir
    ) if annotateDir is not None else [None for _ in range(len(videoPaths))]

    results = {
        "detection_rate": 0.0,
        "mean_track_continuity": 0.0,
    }
    nsamples = 0

    for videoPath, annotatePath in zip(videoPaths, annotatePaths):
        if not videoPath.exists():
            if logging:
                print(f"[{name}] {videoPath.name} does not exist. skipping...")
            continue
        nsamples += 1
        videoRes = _evalVideoYOLO(
            model=model,
            videoPath=videoPath,
            name=name,
            annotatePath=annotatePath,
            colour=colour,
            conf=conf,
            cache=cache,
            cachePath=cachePath,
            logging=logging,
        )
        for key in results.keys():
            if key not in videoRes:
                continue
            results[key] += videoRes[key]

    if nsamples == 0:
        if logging:
            print(f"[{name}] no valid samples in videoPath. returning None...")
        return None
    
    for key in results.keys():
        results[key] /= nsamples
    
    results["model"] = name
    return results

def evaluateVideosFAB(
    model: torch.nn.Module,
    videoPaths: List[Path],
    conf: float=CONFYOLO,
    name: str="footandball",
    annotateDir: Path|None=None,
    colour: Tuple[int, int, int]=COLOURS["footandball"],
    cache: Dict|None=None,
    cachePath: Path|None=None,
    logging: bool=True,
) -> Dict[str, str|float]|None:
    annotatePaths = _getOutputPaths(
        name=name,
        sourcePaths=videoPaths,
        outputDir=annotateDir
    ) if annotateDir is not None else [None for _ in range(len(videoPaths))]

    results = {
        "detection_rate": 0.0,
        "mean_track_continuity": 0.0,
    }
    nsamples = 0

    for videoPath, annotatePath in zip(videoPaths, annotatePaths):
        if not videoPath.exists():
            if logging:
                print(f"[{name}] {videoPath.name} does not exist. skipping...")
            continue
        nsamples += 1
        videoRes = _evalVideoFAB(
            model=model,
            videoPath=videoPath,
            name=name,
            annotatePath=annotatePath,
            colour=colour,
            conf=conf,
            cache=cache,
            cachePath=cachePath,
            logging=logging,
        )
        for key in results.keys():
            if key not in videoRes:
                continue
            results[key] += videoRes[key]

    if nsamples == 0:
        if logging:
            print(f"[{name}] no valid samples in videoPath. returning None...")
        return None
    
    for key in results.keys():
        results[key] /= nsamples
    
    results["model"] = name
    return results


## 7. Evaluate Candidates

In [11]:
def evalModel(
    model: torch.nn.Module|YOLO,
    videoPaths: List[Path],
    name: str,
    annotateDir: Path|None=None,
    colours: Dict[str, Tuple[int, int, int]]=COLOURS,
    cache: Dict|None=None,
    cachePath: Path|None=None,
    logging: bool=True,
) -> Dict[str, str|float]|None:
    if isinstance(model, YOLO):
        return evaluateVideosYOLO(
            model=model,
            videoPaths=videoPaths,
            name=name,
            annotateDir=annotateDir,
            colour=colours[name],
            cache=cache,
            cachePath=cachePath,
            logging=logging,
        )
    return evaluateVideosFAB(
        model=model,
        videoPaths=videoPaths,
        name=name,
        annotateDir=annotateDir,
        colour=colours[name],
        cache=cache,
        cachePath=cachePath,
        logging=logging,
    )

def evaluateCandidates(
    videoPaths: List[Path],
    models: List[Tuple[str, YOLO|torch.nn.Module]],
    annotateDir: Path|None=None,
    colours: Dict[str, Tuple[int, int, int]]=COLOURS,
    resultsPath: Path|None=None,
    cachePath: Path|None=None,
    logging: bool=True
) -> List[Dict[str, str|float]]:
    results = []
    cache = loadJSON(cachePath) or {} if cachePath is not None else None

    for name, model in models:
        print(f"Running model: {name}")
        modelRes = evalModel(
            model=model,
            videoPaths=videoPaths,
            name=name,
            annotateDir=annotateDir,
            colours=colours,
            cache=cache,
            cachePath=cachePath,
            logging=logging,
        )
        if modelRes is None:
            continue
        results.append(modelRes)
    if resultsPath:
        print("saving results...")
        saveJSON(
            data=results,
            path=resultsPath,
        )
        print("saved.")
    return results

results = evaluateCandidates(
    videoPaths=videoPaths,
    models=models,
    annotateDir=annotatedDir,
    colours=COLOURS,
    resultsPath=resultsPath,
    cachePath=resultsCachePath,
    logging=True
)

## 8. Observations

**Experiment Structure:**

- Fine-tuned an `yolo11n` model on Roboflow `football-ball-detection` dataset (v4) for 30 epochs, picked the best weights.
- Evaluated `yolo11n_ft` (fine-tuned model) and `footandball` (pre-trained, broadcast trained model) on several high quality shooting drill footage shot from several different angles.
- Evaluation package consisted of detection rate, mean track continuity and visual inspection from the annotated footage.

Camera angles observed (*{horizontal to subject}-{height}-{distance}*):

*NOTE: low height is ~hip level, mid and high is drone level*
- behind-high-mid
- behind-mid-far
- behind-mid-mid
- behind-low-mid
- behind-low-close
- side-low-far

**Notes:**

- Due to hallucinations and multiple balls in some of the footage, visual inspection was the most reliable evaluation.
- The detection rate gap between models was narrow (0.29 vs 0.22), which alone would suggest the models are closer than they are. This further illustrates why visual inspection was essential here and why detection rate is an unreliable standalone metric for this problem.
- The fine-tuned `yolo11n_ft` performed significantly better than `footandball` on all footage.
- `yolo11n_ft` had "tighter" and more continuous tracking. It was prone to more hallucinations, but for a baseline it was substantially better than `footandball`.
- `footandball` had minimal detections, larger detection zones, and no noticeable continuity.
- Although the objective evaluation metrics were not deemed that reliable, `yolo11n_ft` performed significantly better than `footandball` in both:
  - `yolo11n_ft`: detection rate (~0.29), mean track continuity (~11.61)
  - `footandball`: detection rate (~0.22), mean track continuity (~1.52)
- Both models would benefit from the implementation of a sequence-based approach like in TrackNetV4 ([arXiv:2409.14543](https://arxiv.org/abs/2409.14543)).
- Both models performed best at drone level footage (most similar to the broadcast footage they were trained on) and both degraded significantly as the camera angle lowered. This is the opposite to what was observed for pose models, meaning the two models have opposing failure modes across camera angles. This is not just a model quality issue but a data distribution issue, as neither model has seen close-range training footage.
- Although not officially evaluated, the inference speed of `yolo11n_ft` was significantly faster than `footandball` on Colab GPU T4.

**Decision:**

- `yolo11n_ft` is the chosen baseline ball tracker but more fine-tuning and experimentation is needed before it is a usable model.
- A more problem specific dataset is needed for training and truth labels for the footage are needed for better evaluations.
- From viewing the annotated footage, I believe the model will benefit greatly from a sequence-based approach to tracking (a solution to the jitteriness displayed in some clips).